# Data Inspection
Loads a TWIX scan, extracts k-space, zero-fills it, and displays a preview as an animated GIF. This helps you determine the number of phase-encode lines collected and the offset. Then you can update those values in your configuration file accordingly.

### Loading packages and data

In [2]:
import yaml
import numpy as np
import utils.data_ingestion as di
import utils.gif as gif
from IPython.display import display

def load_config(config_file="config.yaml"):
    """
    Load configuration from a YAML file.

    Parameters
    ----------
    config_file : str
        Path to the YAML configuration file.

    Returns
    -------
    dict
        Parsed configuration data.
    """
    with open(config_file, "r") as f:
        config = yaml.safe_load(f)
    return config

# Read configuration
config = load_config()

# Extract paths and other parameters from config
twix_file = config["data"]["twix_file"]
dicom_folder = config["data"]["dicom_folder"]

# Read TWIX file (only the last scan by default in this example)
scans = di.read_twix_file(twix_file, include_scans=[-1], parse_pmu=False)

Software version: VD/VE (!?)

Scan  1


100%|██████████| 682M/682M [00:00<00:00, 1.02GB/s]

Read 1 scans from new_DATA/raw/meas_MID00049_FID00679_Coronal_Cine_rt_150meas_gre.dat


### Extracting k-space data, zero-filling, and displaying

In [4]:
# Extract raw k-space data
kspace = di.extract_image_data(scans[-1], full_kspace_shape=(150, 128, 30, 256), ref_scan=True)

# Display k-space as an animated GIF
preview = gif.display_kspace_as_gif(kspace, duration=0.2)
display(preview)

Extracted image data shape: (150, 128, 30, 256)


In [9]:
from utils.reconstruction import grappa_reconstruction

# Figure out which lines are acquired
acquired_lines = np.where(~np.all(kspace == 0, axis=(0, 2, 3)))[0]
# Example: acquired_lines = [0, 2, 4, 6, 7, 8, 9, 10, 12, 14, 16]

# Figure out the largest contiguous block of acquired lines
contiguous_lines = np.split(acquired_lines, np.where(np.diff(acquired_lines) != 1)[0] + 1)
largest_block = max(contiguous_lines, key=len)
smallest_line, largest_line = largest_block[0], largest_block[-1]
print(f"Smallest line: {smallest_line}, Largest line: {largest_line}")

print("Performing GRAPPA...")
images = grappa_reconstruction(kspace[:10, ...], calib_region=(smallest_line, largest_line))
print("Reconstruction complete.")
images = np.rot90(images, k=1, axes=(1, 2))
images = np.flip(images, axis=2)
images = images[:, 64:-64, :]
preview = gif.display_images_as_gif(images, notebook=True)
display(preview)

Smallest line: 52, Largest line: 76
Performing GRAPPA...
Reconstruction complete.


In [4]:
# # Extract raw k-space data
# kspace = di.extract_image_data(scans[-1])

# n_frames = di.get_num_frames(dicom_folder)
# n_coils = kspace.shape[1]

# # Reshape k-space into frames
# kspace = np.reshape(kspace, (n_frames, -1, n_coils, kspace.shape[2]))

# # Pull the total number of phase encodes and define offset
# extended_pe_lines = di.get_total_phase_encodes(dicom_folder)
# offset = 32  # Adjust if needed

# # Allocate zero-filled array
# kspace_zf = np.zeros((n_frames, extended_pe_lines, n_coils, kspace.shape[3]), dtype=np.complex64)
# kspace_zf[:, offset : offset + kspace.shape[1], :] = kspace

# # Display k-space as an animated GIF
# preview = gif.display_kspace_as_gif(kspace_zf, duration=0.2)
# display(preview)

In [6]:
# from utils.reconstruction import direct_ifft_reconstruction
# images = direct_ifft_reconstruction(kspace, extended_pe_lines, 0, use_conjugate_symmetry=False)
# images = np.rot90(images, k=1, axes=(1, 2))
# images = np.flip(images, axis=2)
# images = images[:, 64:-64, :]
# preview = gif.display_images_as_gif(images, notebook=True)
# display(preview)

In [ ]:
for (i,mdb) in enumerate(scans[-1]['mdb']):
    if mdb.is_image_scan():
        print(
            mdb.cLin, # Specific line
            mdb.cRep, # Frame
            mdb.cSeg, # Segment
        )
    elif mdb.is_flag_set('PATREFSCAN'):
        print(
            mdb.cLin, # Specific line
            mdb.cRep, # Frame
            mdb.cSeg, # Segment
            "REFERENCE"
        )